### Simple GENAI App Using OpenAI (GenAIAppWithOpenAI)
- Selecting a website, scrap it's content using data ingestion
- Split the data
- Convert it into vectors, maybe store into VectorDB
- Use LLM along with Prompt Engineering to get an output

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## For LangSmith Tracking
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT")
 
# Usually optional for hosted LangSmith, but safe to set
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")

In [4]:
# Data Ingestion -- From the website
from langchain_community.document_loaders import WebBaseLoader
    
loader = WebBaseLoader("https://whc.unesco.org/en/list/232/")
loader


In [5]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://whc.unesco.org/en/list/232/', 'title': "Humayun's Tomb, Delhi - UNESCO World Heritage Centre", 'description': 'UNESCO World Heritage Centre', 'language': 'en'}, page_content="\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nHumayun's Tomb, Delhi - UNESCO World Heritage Centre\n\n\n\n\nYour browser does not support JavaScript.\n\n\n\n\n\n\n\r\n                    World Heritage Convention\r\n                \n\n\n\n\n\n\n\n\n\nHelp preserve sites now!\n\n\nExplore UNESCO\n\n\n\nEnglish\n\n\n\nEnglish\n\n\nFrançais\n\n\n\n\n\n\nLogin\n\n\n  Our expertise    About World Heritage News Events The Convention Convention Text Basic Texts Operational Guidelines Policy Compendium Declaration of principles to promote international solidarity and cooperation to preserve World Heritage   Governing Bodies The General Assembly The Committee Resolutions / Decisions Sessions since 1976 The States Parties The Advisory Bodies  

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://whc.unesco.org/en/list/232/', 'title': "Humayun's Tomb, Delhi - UNESCO World Heritage Centre", 'description': 'UNESCO World Heritage Centre', 'language': 'en'}, page_content="Humayun's Tomb, Delhi - UNESCO World Heritage Centre\n\n\n\n\nYour browser does not support JavaScript.\n\n\n\n\n\n\n\r\n                    World Heritage Convention\r\n                \n\n\n\n\n\n\n\n\n\nHelp preserve sites now!\n\n\nExplore UNESCO\n\n\n\nEnglish\n\n\n\nEnglish\n\n\nFrançais\n\n\n\n\n\n\nLogin"),
 Document(metadata={'source': 'https://whc.unesco.org/en/list/232/', 'title': "Humayun's Tomb, Delhi - UNESCO World Heritage Centre", 'description': 'UNESCO World Heritage Centre', 'language': 'en'}, page_content="Our expertise    About World Heritage News Events The Convention Convention Text Basic Texts Operational Guidelines Policy Compendium Declaration of principles to promote international solidarity and cooperation to preserve World Heritage   Governing Bodi

In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()
embedding_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x11f51fdc0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x11f51ff10>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [9]:
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(documents, embedding_model)
vectorstoredb

In [10]:
query = "Humayun’s Tomb, Delhi is the first of the grand dynastic mausoleums"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'Outstanding Universal Value\nBrief Synthesis\nHumayun’s Tomb, Delhi is the first of the grand dynastic mausoleums that were to become synonyms of Mughal architecture with the architectural style reaching its zenith 80 years later at the later Taj Mahal. Humayun’s Tomb stands within a complex of 27.04 ha. that includes other contemporary, 16th century Mughal garden-tombs such as Nila Gumbad, Isa Khan, Bu Halima, Afsarwala, Barber’s Tomb and the complex where the craftsmen employed for the Building of Humayun’s Tomb stayed, the Arab Serai.'

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x106c93e20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11f8014b0>, root_client=<openai.OpenAI object at 0x11f55c3d0>, root_async_client=<openai.AsyncOpenAI object at 0x11f800e80>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [12]:
## Document Chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        Answer the following question based on the provided context
        <context>
            {context}
        </context>
        
        <input>
            {input}
        </input>
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following question based on the provided context\n        <context>\n            {context}\n        </context>\n        \n        <input>\n            {input}\n        </input>\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x106c93e20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x11f8014b0>, root_client=<openai.OpenAI object at 0x11f55c3d0>, root_async_client=<openai.AsyncOpenAI object at 0x11f800e80>, model_name='gpt-4o', model_kw

In [13]:
## What basically Document chain is doing is providing the context to the prompt and providing that prompt to the llm along with our input. PLEASE NOTE THAT HERE THE CONTEXT IS PROVIDED MANUALLY, IN THE NEXT CELL, WE MAKE USE OF THE RETRIEVERS TO RETRIEVE THE DOCS DYNAMICALLY AND USE IN THE PROMPT

from langchain_core.documents import Document

document_chain.invoke({
    "input": "Humayun’s Tomb, Delhi is the first of the grand dynastic mausoleums",
    "context": [Document(page_content="Humayun’s Tomb, Delhi is the first of the grand dynastic mausoleums that were to become synonyms of Mughal architecture with the architectural style reaching its zenith 80 years later at the later Taj Mahal. Humayun’s Tomb stands within a complex of 27.04 ha. that includes other contemporary, 16th century Mughal garden-tombs such as Nila Gumbad, Isa Khan, Bu Halima, Afsarwala, Barber’s Tomb and the complex where the craftsmen employed for the Building of Humayun’s Tomb stayed, the Arab Serai.")]
})

'Humayun’s Tomb in Delhi is indeed recognized as the first of the grand dynastic mausoleums that became synonymous with Mughal architecture. This architectural style reached its zenith 80 years later with the construction of the Taj Mahal. Humayun’s Tomb is part of a larger complex spanning 27.04 hectares, including other 16th-century Mughal garden-tombs like Nila Gumbad, Isa Khan, Bu Halima, Afsarwala, Barber’s Tomb, and the area known as the Arab Serai, where craftsmen who built Humayun’s Tomb resided.'

However we want the documents to come from the RETRIEVER, that way we can use the RETRIEVER to select the most relevant documents and pass those for the given question.

In [14]:
## Retriever --> responsible for retrieving the most relevant documents with respect to the user input, from the DB
retriever = vectorstoredb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11fa9f070>, search_kwargs={})

In [15]:
## Retrieval Chain
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11fa9f070>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following question based on the provided context\n        <context>\n            {context}\n        </context>\n     

In [16]:
## Get response from the LLM
response = retrieval_chain.invoke({
    "input" : "What is Humayun's tomb?"
})
response["answer"]

"Humayun's Tomb is a grand dynastic mausoleum in Delhi, India, and the first significant example of Mughal architecture in the country. It was built in the 1560s under the patronage of Humayun's son, Emperor Akbar. The tomb is part of a complex that spans 27.04 hectares and includes other 16th-century Mughal garden-tombs such as Nila Gumbad, Isa Khan, and others. It is an important architectural innovation featuring a charbagh, a garden that represents the Quranic paradise, with pools and flowing channels. Humayun's Tomb is also called the ‘dormitory of the Mughals,’ as it houses over 150 Mughal family members. The tomb is an emblem of the powerful Mughal dynasty and stands in close proximity to the Shrine of the 14th-century Sufi Saint, Hazrat Nizamuddin Auliya, in a significant archaeological setting. This model of garden-tomb architecture was later epitomized by the Taj Mahal."

In [17]:
response

{'input': "What is Humayun's tomb?",
 'context': [Document(id='d10afa7c-284c-4444-99a6-013d97716b18', metadata={'source': 'https://whc.unesco.org/en/list/232/', 'title': "Humayun's Tomb, Delhi - UNESCO World Heritage Centre", 'description': 'UNESCO World Heritage Centre', 'language': 'en'}, page_content='Criteria (iv):\xa0Humayun’s Tomb and the other contemporary 16th century garden tombs within the property form a unique ensemble of Mughal era garden-tombs. The monumental scale, architectural treatment and garden setting are outstanding in Islamic garden-tombs. Humayun’s Tomb is the first important example in India, and above all else, the symbol of the powerful Mughal dynasty that unified most of the sub continent.\nIntegrity'),
  Document(id='0586ebd7-c2ae-441e-aee8-e24b93c71715', metadata={'source': 'https://whc.unesco.org/en/list/232/', 'title': "Humayun's Tomb, Delhi - UNESCO World Heritage Centre", 'description': 'UNESCO World Heritage Centre', 'language': 'en'}, page_content='O